In [0]:
# Debug Cell — run this before anything else
bronze = spark.table("default.bronze_retail")

print("=== Bronze columns ===")
print(bronze.columns)

print("\n=== First 3 rows raw ===")
bronze.show(3, truncate=False)

print("\n=== Schema ===")
bronze.printSchema()

In [0]:
from pyspark.sql.functions import *

bronze = spark.table("default.bronze_retail")

silver = (bronze
    .filter(~col("InvoiceNo").startswith("C"))
    .filter(col("CustomerID").isNotNull())
    .filter(col("Description").isNotNull())

    .withColumn("invoice_date",
        to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
    .withColumn("customer_id",
        col("CustomerID").cast("int"))
    .withColumn("quantity",
        col("Quantity").cast("int"))
    .withColumn("unit_price",
        col("UnitPrice").cast("double"))

    .filter(col("quantity")   > 0)
    .filter(col("unit_price") > 0.0)

    .dropDuplicates(["InvoiceNo", "StockCode"])

    .withColumnRenamed("InvoiceNo",   "invoice_no")
    .withColumnRenamed("StockCode",   "stock_code")
    .withColumnRenamed("Description", "description")
    .withColumnRenamed("Country",     "country")

    .withColumn("revenue",
        round(col("quantity") * col("unit_price"), 2))
    .withColumn("invoice_month",
        date_format("invoice_date", "yyyy-MM"))
    .withColumn("invoice_year",  year("invoice_date"))
    .withColumn("day_of_week",   dayofweek("invoice_date"))

    .filter(col("revenue") > 0)

    # ── "Quantity" removed from here — withColumn already replaced it ──
    .drop("InvoiceDate", "CustomerID", "UnitPrice",
          "_ingested_at", "_source")

    .withColumn("_processed_at", current_timestamp())
)

silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_retail")

print(f"Silver rows written: {silver.count():,}")
print(f"Columns: {spark.table('default.silver_retail').columns}")

In [0]:
s = spark.table("default.silver_retail")

checks = {
    "no null customer_id":  s.filter(col("customer_id").isNull()).count() == 0,
    "no negative revenue":  s.filter(col("revenue") <= 0).count() == 0,
    "no null invoice_date": s.filter(col("invoice_date").isNull()).count() == 0,
    "no cancellations":     s.filter(col("invoice_no").startswith("C")).count() == 0,
}

for check, passed in checks.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}  {check}")

assert all(checks.values()), "Quality checks failed — stopping pipeline"

In [0]:
count = spark.table("default.silver_retail").count()
dbutils.notebook.exit(f"silver_rows={count:,}")